In [ ]:
# First, we read csv file
import pandas as pd
import ast
df = pd.read_csv("Data.csv", sep=";")


df["GPT_ID1"] = df["GPT_ID1"].apply(ast.literal_eval)
df["GPT_ID2"] = df["GPT_ID2"].apply(ast.literal_eval)
df["E5_ID1"] = df["E5_ID1"].apply(ast.literal_eval)
df["E5_ID2"] = df["E5_ID2"].apply(ast.literal_eval)
df["lex_overlap"] = pd.to_numeric(df["lex_overlap"])

In [ ]:
import torch
from transformers import GPT2Model, GPT2Tokenizer, AutoModel, AutoTokenizer

# Load tokenizers and models
GPTtokenizer = GPT2Tokenizer.from_pretrained("gpt2")
GPTmodel = GPT2Model.from_pretrained("gpt2")
E5tokenizer = AutoTokenizer.from_pretrained('intfloat/e5-small-v2')
E5model = AutoModel.from_pretrained('intfloat/e5-small-v2')

***Here we calculate VLDDR and Centroid***

In [ ]:
def chordal_distance(v1, v2):
    norm_v1 = torch.nn.functional.normalize(v1, dim = 0)
    norm_v2 = torch.nn.functional.normalize(v2, dim = 0)
    return float(torch.norm(norm_v1 - norm_v2, p=2))

def calc_non_context_dist(a, b):
    non_contextual1 = embedder(a)
    non_contextual2 = embedder(b)
    x = non_contextual1.mean(dim=0)
    y = non_contextual2.mean(dim=0)
    return(chordal_distance(x,y))


#First, we calculate for GPT 2
embedder = GPTmodel.get_input_embeddings()

df["non_contextual_dist_GPT"] = df.apply(
    lambda row: calc_non_context_dist(torch.tensor(row["GPT_ID1"]), torch.tensor(row["GPT_ID2"])),
    axis=1
)

#Then we calculate for E5
embedder = E5model.get_input_embeddings()

df["non_contextual_dist_E5"] = df.apply(
    lambda row: calc_non_context_dist(torch.tensor(row["E5_ID1"]), torch.tensor(row["E5_ID2"])),
    axis=1
)

The following ran in about ~10 minutes on an AMD Ryzen 5 2600 six core CPU

In [ ]:
# Contextual embeddings difference
def calc_context_dist(a, b):
    with torch.no_grad():
        output1 = model(a)
        output2 = model(b)

    contextual1 = output1.last_hidden_state
    contextual2 = output2.last_hidden_state

    contextual1 = contextual1.squeeze(0)
    contextual2 = contextual2.squeeze(0)

    x = contextual1.mean(dim=0)
    y = contextual2.mean(dim=0)
    
    return(chordal_distance(x,y))

# First, We calculate for GPT 2
model = GPTmodel

df["contextual_dist_GPT"] = df.apply(
    lambda row: calc_context_dist(torch.tensor(row["GPT_ID1"]), torch.tensor(row["GPT_ID2"])),
    axis=1
)

# Then, for E5
def get_E5_embeddings(text):
    encoded = E5tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        output = E5model(**encoded)
    
    embeddings = output.last_hidden_state.squeeze(0)[1:-1]
    return embeddings


def calc_context_dist(a, b):
    contextual1 = get_E5_embeddings(a)
    contextual2 = get_E5_embeddings(b)

    x = contextual1.mean(dim=0)
    y = contextual2.mean(dim=0)
    
    return(chordal_distance(x,y))

df["contextual_dist_E5"] = df.apply(
    lambda row: calc_context_dist(row["sentence1"], row["sentence2"]),
    axis=1
)

The following ran in about ~10 minutes on an AMD Ryzen 5 2600 six core CPU

In [ ]:
# Centroid + cosine sim distance
def cosine_sim(v1, v2):
    norm_v1 = torch.nn.functional.normalize(v1, dim = 0)
    norm_v2 = torch.nn.functional.normalize(v2, dim = 0)
    return float(torch.dot(norm_v1, norm_v2))


def calc_centroid(a, b):
    with torch.no_grad():
        output1 = model(a)
        output2 = model(b)

    contextual1 = output1.last_hidden_state
    contextual2 = output2.last_hidden_state

    contextual1 = contextual1.squeeze(0)
    contextual2 = contextual2.squeeze(0)

    x = contextual1.mean(dim=0)
    y = contextual2.mean(dim=0)
    
    return(cosine_sim(x,y))

# First, we calculate for GPT
model = GPTmodel

df["Centroid_GPT"] = df.apply(
    lambda row: calc_centroid(torch.tensor(row["GPT_ID1"]), torch.tensor(row["GPT_ID2"])),
    axis=1
)

# Then, we calculate for E5
def calc_centroid(a, b):
    contextual1 = get_E5_embeddings(a)
    contextual2 = get_E5_embeddings(b)

    x = contextual1.mean(dim=0)
    y = contextual2.mean(dim=0)
    
    return(cosine_sim(x,y))

df["Centroid_E5"] = df.apply(
    lambda row: calc_centroid(row["sentence1"], row["sentence2"]),
    axis=1
)

In [ ]:
df["VLDDR_GPT"] = df["non_contextual_dist_GPT"]/df["contextual_dist_GPT"]
df["VLDDR_E5"] = df["non_contextual_dist_E5"]/df["contextual_dist_E5"]

In [ ]:
df.to_csv("VLDDR_Centroid.csv", sep=";", index=False)